# Data for “Patriarchal Norms in Religious Institutional Discourses: a feminist critical discourse analysis of responses to conflict-related sexual violence in Bosnia and Herzegovina” Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

This notebook emphasizes a metadata-first workflow because Croissant datasets may expose multiple record sets, fields, file objects, and distributions. Per Croissant best practices, all dataset entities should be referenced by their `@id` values whenever they are available.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In this section, we:
- create a dataset object from the Croissant JSON-LD URL,
- inspect the dataset metadata as a single object,
- convert metadata to JSON only when we need a serializable representation for inspection.

In [ ]:
import json
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

# Define the dataset URL
url = "https://sen.science/doi/10.71728/senscience.bhyq-171e/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_obj = dataset.metadata
metadata_json = metadata_obj.to_json()

print("Dataset loaded successfully.")
print(f"Name: {getattr(metadata_obj, 'name', 'N/A')}")
print(f"Description: {getattr(metadata_obj, 'description', 'N/A')}")
print(f"Identifier: {metadata_json.get('identifier', 'N/A')}")
print(f"License: {getattr(metadata_obj, 'license', 'N/A')}")
print(f"Version: {getattr(metadata_obj, 'version', 'N/A')}")

In [ ]:
# Optional: inspect the top-level metadata JSON keys
top_level_keys = sorted(metadata_json.keys())
print("Top-level metadata keys:")
print(top_level_keys)

## 2. Data Overview
Review available record sets, fields, and their IDs.

The FAIR^2 package shown in the prompt indicates that `recordSet` is empty in the provided metadata snapshot. Since Croissant exploration should still be systematic, we first inspect the metadata to confirm whether record sets are present at runtime.

If record sets exist, we will enumerate them and inspect sample records using their `@id` values. If not, the notebook will gracefully fall back to metadata exploration.

In [ ]:
record_sets_metadata = metadata_json.get("recordSet", []) or []
print(f"Number of record sets declared in metadata JSON: {len(record_sets_metadata)}")

if record_sets_metadata:
    overview_rows = []
    for rs in record_sets_metadata:
        rs_id = rs.get("@id", "N/A")
        rs_name = rs.get("name", "N/A")
        rs_desc = rs.get("description", "")
        field_count = len(rs.get("field", []) or [])
        overview_rows.append({
            "record_set_id": rs_id,
            "record_set_name": rs_name,
            "field_count": field_count,
            "description": rs_desc,
        })
    record_sets_df = pd.DataFrame(overview_rows)
    display(record_sets_df)
else:
    print("No record sets were declared in the Croissant metadata snapshot.")

In [ ]:
# Inspect field-level metadata, always referenced by @id when available
if record_sets_metadata:
    field_rows = []
    for rs in record_sets_metadata:
        rs_id = rs.get("@id", "N/A")
        for field in rs.get("field", []) or []:
            field_rows.append({
                "record_set_id": rs_id,
                "field_id": field.get("@id", "N/A"),
                "field_name": field.get("name", "N/A"),
                "data_type": field.get("dataType", "N/A"),
                "source": field.get("source", "N/A"),
                "description": field.get("description", ""),
            })
    fields_df = pd.DataFrame(field_rows)
    display(fields_df)
else:
    fields_df = pd.DataFrame()
    print("No field metadata is available because no record sets were declared.")

In [ ]:
# Try to preview records for each record set using the record set @id.
# This cell is defensive: it handles cases where there are no record sets.
if record_sets_metadata:
    for rs in record_sets_metadata:
        record_set_id = rs.get("@id")
        print(f"\nPreviewing records for record set @id: {record_set_id}")
        try:
            preview = []
            for i, x in enumerate(dataset.records(record_set=record_set_id)):
                preview.append(x)
                if i >= 2:
                    break
            if preview:
                for item in preview:
                    print(item)
            else:
                print("No records returned for this record set.")
        except Exception as e:
            print(f"Unable to read records for {record_set_id}: {e}")
else:
    print("Skipping record preview because no record set @id values are available.")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use record set and field `@id` values from the overview.

Because this dataset may expose zero or more record sets depending on the Croissant source, the extraction logic below is dynamic and robust:
- it collects record set `@id` values from metadata,
- loads each record set into a pandas DataFrame,
- stores all DataFrames in a dictionary keyed by the record set `@id`.

In [ ]:
# Extract data from each record set
record_set_ids = [rs.get("@id") for rs in record_sets_metadata if rs.get("@id")]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records from {record_set_id}")
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")

if dataframes:
    first_record_set_id = next(iter(dataframes))
    print(f"\nFirst available record set @id: {first_record_set_id}")
    print("Columns:")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print("No tabular record sets were loaded. We will continue with metadata-driven exploration.")

Even when tabular record sets are absent, Croissant metadata itself is highly informative. The next cell organizes selected top-level metadata into compact tables for downstream analysis.

In [ ]:
# Build metadata-driven tables for analysis when record sets are missing or sparse
scalar_metadata = {
    "dataset_id": metadata_json.get("@id"),
    "identifier": metadata_json.get("identifier"),
    "name": metadata_json.get("name"),
    "description": metadata_json.get("description"),
    "license": metadata_json.get("license"),
    "version": metadata_json.get("version"),
    "date_published": metadata_json.get("datePublished"),
    "spatial_coverage": metadata_json.get("spatialCoverage"),
    "temporal_coverage": metadata_json.get("temporalCoverage"),
    "cite_as": metadata_json.get("citeAs"),
}

dataset_summary_df = pd.DataFrame([scalar_metadata])
display(dataset_summary_df.T.rename(columns={0: 'value'}))

list_fields = [
    "keywords",
    "dataUseCases",
    "dataBiases",
    "dataLimitations",
    "dataPreprocessingProtocol",
    "dataCollectionTimeframe",
    "dataCollectionType",
    "personalSensitiveInformation",
]

metadata_tables = {}
for field_id in list_fields:
    values = metadata_json.get(field_id, [])
    if not isinstance(values, list):
        values = [values]
    metadata_tables[field_id] = pd.DataFrame({field_id: values})
    print(f"\nField: {field_id}")
    display(metadata_tables[field_id])

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Since the supplied metadata snapshot does not declare record sets, this section demonstrates two complementary EDA styles:
1. **Tabular record-set EDA**, if any record set with numeric columns is available at runtime.
2. **Metadata EDA**, using the dataset's descriptive lists such as keywords, biases, and use cases.

This keeps the notebook faithful to the template while remaining executable for metadata-centric Croissant packages.

In [ ]:
# Attempt record-set-based EDA using numeric fields, if available
numeric_eda_completed = False

if dataframes:
    for record_set_id, df in dataframes.items():
        numeric_columns = df.select_dtypes(include="number").columns.tolist()
        if numeric_columns:
            numeric_field = numeric_columns[0]
            working_df = df.copy()
            threshold = working_df[numeric_field].median()
            filtered_df = working_df[working_df[numeric_field] > threshold].copy()
            print(f"Using record set @id: {record_set_id}")
            print(f"Selected numeric field: {numeric_field}")
            print(f"Threshold (median): {threshold}")
            print(f"Filtered records with {numeric_field} > {threshold}:")
            display(filtered_df.head())

            std_value = filtered_df[numeric_field].std()
            if pd.notna(std_value) and std_value != 0:
                filtered_df[f"{numeric_field}_normalized"] = (
                    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
                )
            else:
                filtered_df[f"{numeric_field}_normalized"] = 0.0

            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            group_candidates = [col for col in df.columns if df[col].dtype == 'object']
            if group_candidates:
                group_field = group_candidates[0]
                print(f"Grouping by: {group_field}")
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean(numeric_only=False).reset_index()
                display(grouped_df.head())
            else:
                print("No categorical/object field available for grouping in this record set.")

            numeric_eda_completed = True
            break

if not numeric_eda_completed:
    print("No numeric record-set columns available. Proceeding with metadata-driven EDA instead.")

In [ ]:
# Metadata-driven EDA: create a unified table of descriptive list items
analysis_fields = {
    "keywords": metadata_json.get("keywords", []),
    "dataUseCases": metadata_json.get("dataUseCases", []),
    "dataBiases": metadata_json.get("dataBiases", []),
    "dataLimitations": metadata_json.get("dataLimitations", []),
    "dataCollectionType": metadata_json.get("dataCollectionType", []),
    "personalSensitiveInformation": metadata_json.get("personalSensitiveInformation", []),
}

rows = []
for field_id, values in analysis_fields.items():
    if not isinstance(values, list):
        values = [values]
    for idx, value in enumerate(values):
        rows.append({
            "source_field_id": field_id,
            "item_index": idx,
            "text": value,
            "text_length": len(str(value)),
            "word_count": len(str(value).split()),
        })

metadata_items_df = pd.DataFrame(rows)
display(metadata_items_df.head(10))

# Select a numeric field for analysis
numeric_field = "word_count"
threshold = 5
filtered_df = metadata_items_df[metadata_items_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

std_value = filtered_df[numeric_field].std()
if pd.notna(std_value) and std_value != 0:
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
    )
else:
    filtered_df[f"{numeric_field}_normalized"] = 0.0

print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

group_field = "source_field_id"
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[["text_length", "word_count"]].mean().sort_values("word_count", ascending=False)
    print(f"Grouped data by {group_field}:")
    display(grouped_df)

A useful extension is lightweight text categorization. Below, we derive simple labels from metadata items to distinguish between shorter tags and more narrative statements.

In [ ]:
metadata_items_df = metadata_items_df.copy()
metadata_items_df["length_category"] = pd.cut(
    metadata_items_df["word_count"],
    bins=[-1, 3, 10, 1000],
    labels=["short", "medium", "long"]
)

category_summary_df = (
    metadata_items_df.groupby(["source_field_id", "length_category"])
    .size()
    .reset_index(name="count")
    .sort_values(["source_field_id", "count"], ascending=[True, False])
)

display(category_summary_df)

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We focus on metadata-derived visualizations because the provided Croissant snapshot is metadata-rich and does not explicitly declare record sets.

In [ ]:
# Visualization 1: number of items per metadata field
field_counts_df = (
    metadata_items_df.groupby("source_field_id")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

plt.figure(figsize=(10, 5))
sns.barplot(data=field_counts_df, x="count", y="source_field_id", palette="viridis")
plt.title("Count of Items by Metadata Field ID")
plt.xlabel("Count")
plt.ylabel("Metadata field @id/key")
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: distribution of word counts across metadata text items
plt.figure(figsize=(8, 5))
sns.histplot(metadata_items_df["word_count"], bins=15, kde=True, color="steelblue")
plt.title("Distribution of Metadata Text Word Counts")
plt.xlabel("Word count")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 3: average text length by metadata field
avg_length_df = (
    metadata_items_df.groupby("source_field_id")[["text_length", "word_count"]]
    .mean()
    .reset_index()
    .sort_values("word_count", ascending=False)
)

plt.figure(figsize=(10, 5))
sns.barplot(data=avg_length_df, x="word_count", y="source_field_id", palette="magma")
plt.title("Average Word Count by Metadata Field ID")
plt.xlabel("Average word count")
plt.ylabel("Metadata field @id/key")
plt.tight_layout()
plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This exploration shows how to work with a Croissant dataset using `mlcroissant` even when the package is primarily metadata-oriented.

Key observations:
- The dataset provides rich documentation about provenance, collection procedures, preprocessing, limitations, and responsible use.
- In the supplied metadata snapshot, no `recordSet` entries are declared, so the most reliable analysis path is metadata-driven rather than table-driven.
- The metadata indicates a very small, curated, and sensitive archival corpus related to conflict-related sexual violence and religious institutional discourse in Bosnia and Herzegovina.
- Important usage considerations include limited corpus size, translation effects, archival access constraints, and the need for careful contextual interpretation.

If future versions of the Croissant package expose record sets, the same notebook structure can immediately support record extraction by record set `@id`, field inspection by field `@id`, and more conventional tabular EDA.